In [1]:
import os
from pathlib import Path

DATA_ROOT = Path.home() / "mne_data"

os.environ["MNE_DATA"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_BNCI_PATH"] = str(DATA_ROOT)
os.environ["MNE_DATASETS_ZHOU2016_PATH"] = str(DATA_ROOT)
os.environ["MOABB_DOWNLOAD_PROVIDER"] = "upstream"

EXPERIMENT_DIR = Path("/Users/liza/Projects/vkr/mi-ood")
CACHE_DIR = EXPERIMENT_DIR / "cache"
RESULTS_DIR = EXPERIMENT_DIR / "results"

print(EXPERIMENT_DIR)

/Users/liza/Projects/vkr/mi-ood


In [2]:
from pipelines import mi_pipelines, ood_pipelines
from processing import *

/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/optuna/integration/sklearn.py:14: FutureWarning: `optuna.integration.sklearn` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.sklearn` instead.
  optuna_warn(f"{msg} Use `optuna_integration.sklearn` instead.", FutureWarning)
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:497: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage("standard_1020")
/Users/liza/Projects/bci-motor-imagery/.venv/lib/python3.11/site-packages/braindecode/models/eegpt.py:1452: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = make_standard_montage("standard_1020")


In [3]:
experiment = "cross-session_7-channels-rest"

COMMON_CHANNELS = ["FC3", "FCz", "FC4", "C3", "C4", "CP3", "CP4"]

datasets = [ "Zhou2016", "BNCI2014_001", "TTK", "Zhou2020", "Wang2026_45", "Stieger2021_fix" ] # 4 9 6 20, 39, 62


MI_METHODS = ['ACM+TS+SVM', 'FBCSP+ElasticNet', 'ShallowConvNet', 'EEGNet', "Microstates+GB", "VAR+shLDA", "Log-bandpower+shLDA" ]
OOD_METHODS = ['ACM+TS+OCSVM', 'FBCSP+IsolationForest', 'ShallowConvNet+kNN', 'EEGNet+TempDens', "Log-bandpower+Mahalanobis", "Microstates+LOF", "VAR+Mahalanobis" ]


In [4]:
for dataset in datasets:
    print( dataset )
    subjects = get_subjects(dataset) 

    for subject in subjects:
        data = None
        
        for method, pipeline in mi_pipelines.items():
            if method not in MI_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": "mi", "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, duration=3.0, print_count=True)
            train_data, val_data, test_data = data
            result["MI_BA"], result["MI_AUROC"] = MI_metrics(train_data, test_data, pipeline)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['MI_BA']:.3f} {result['MI_AUROC']:.3f}")

        data = None
        for method, pipeline in ood_pipelines.items():
            if method not in OOD_METHODS:
                continue
            result = {"dataset": dataset, "subject": subject, "experiment": experiment, "experiment_type": "ood", "method": method}
            if result_path(result).exists():
                continue
            if data is None:
                data = load_subject_data(dataset, subject, channels=COMMON_CHANNELS, ood_classes=("rest",), duration=3.0, print_count=True )
                train_data, val_data, test_data = data

            model = train(train_data, pipeline)

            result["OOD_AUROC"] = OOD_AUROC(test_data, model)
            save_result(result)
            print(f"{dataset} s{subject} {method}: {result['OOD_AUROC']:.3f}")

    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_BA")
    display_summary_by_subject(dataset, experiment, experiment_type="mi", metric="MI_AUROC")
    display_summary_by_subject(dataset, experiment, experiment_type="ood", metric="OOD_AUROC")

Zhou2016


,MI_BA,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
EEGNet,0.847,4,0.810,0.778,0.910,0.890
ShallowConvNet,0.784,4,0.700,0.678,0.890,0.870
FBCSP+ElasticNet,0.774,4,0.690,0.644,0.890,0.870
ACM+TS+SVM,0.738,4,0.610,0.633,0.790,0.920
Microstates+GB,0.717,4,0.530,0.589,0.840,0.910
Log-bandpower+shLDA,0.697,4,0.640,0.689,0.810,0.650
VAR+shLDA,0.641,4,0.680,0.633,0.560,0.690


,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
EEGNet,0.921,4,0.893,0.841,0.973,0.978
ShallowConvNet,0.891,4,0.875,0.766,0.978,0.945
FBCSP+ElasticNet,0.875,4,0.870,0.746,0.947,0.936
ACM+TS+SVM,0.858,4,0.810,0.719,0.927,0.978
Log-bandpower+shLDA,0.805,4,0.762,0.755,0.914,0.787
Microstates+GB,0.780,4,0.590,0.664,0.901,0.966
VAR+shLDA,0.681,4,0.677,0.731,0.563,0.753


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04
method,,,,,,
EEGNet+TempDens,0.771,4,0.902,0.871,0.724,0.588
ShallowConvNet+kNN,0.728,4,0.716,0.907,0.654,0.634
ACM+TS+OCSVM,0.714,4,0.823,0.868,0.598,0.566
FBCSP+IsolationForest,0.712,4,0.848,0.835,0.727,0.436
Log-bandpower+Mahalanobis,0.644,4,0.758,0.793,0.627,0.397
VAR+Mahalanobis,0.641,4,0.702,0.801,0.568,0.494
Microstates+LOF,0.620,4,0.626,0.791,0.544,0.518


BNCI2014_001


,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09
method,,,,,,,,,,,
EEGNet,0.781,9,0.889,0.576,0.958,0.778,0.576,0.743,0.632,0.972,0.903
ACM+TS+SVM,0.752,9,0.826,0.590,0.938,0.722,0.639,0.722,0.576,0.986,0.764
FBCSP+ElasticNet,0.750,9,0.840,0.604,0.958,0.590,0.660,0.681,0.688,0.958,0.771
ShallowConvNet,0.731,9,0.819,0.514,0.896,0.667,0.569,0.715,0.597,0.944,0.854
Microstates+GB,0.677,9,0.785,0.493,0.854,0.597,0.556,0.639,0.542,0.917,0.715
Log-bandpower+shLDA,0.653,9,0.653,0.500,0.812,0.576,0.549,0.625,0.590,0.785,0.785
VAR+shLDA,0.649,9,0.646,0.535,0.840,0.549,0.597,0.646,0.569,0.715,0.743


... (14 outputs left out)

Stieger2021_fix


,MI_BA,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20,sub-21,sub-22,sub-23,sub-24,sub-25,sub-26,sub-27,sub-28,sub-29,sub-30,sub-31,sub-32,sub-33,sub-34,sub-35,sub-36,sub-37,sub-38,sub-39,sub-40,sub-41,sub-42,sub-43,sub-44,sub-45,sub-46,sub-47,sub-48,sub-49,sub-50,sub-51,sub-52,sub-53,sub-54,sub-55,sub-56,sub-57,sub-58,sub-59,sub-60,sub-61,sub-62
method,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FBCSP+ElasticNet,0.624,62,0.796,0.583,0.535,0.833,0.744,0.607,0.488,0.629,0.702,0.558,0.534,0.680,0.542,0.544,0.544,0.576,0.503,0.565,0.750,0.585,0.553,0.589,0.530,0.808,0.660,0.592,0.562,0.959,0.827,0.799,0.491,0.501,0.569,0.500,0.712,0.626,0.802,0.783,0.763,0.606,0.545,0.477,0.561,0.432,0.697,0.707,0.548,0.481,0.686,0.887,0.500,0.648,0.656,0.668,0.493,0.677,0.814,0.555,0.500,0.450,0.578,0.620
EEGNet,0.615,62,0.768,0.715,0.418,0.750,0.706,0.641,0.565,0.803,0.696,0.533,0.527,0.570,0.533,0.554,0.392,0.538,0.496,0.566,0.778,0.800,0.626,0.500,0.692,0.712,0.706,0.655,0.480,0.932,0.783,0.813,0.544,0.605,0.473,0.513,0.740,0.662,0.846,0.636,0.625,0.558,0.510,0.456,0.559,0.514,0.542,0.778,0.557,0.631,0.442,0.887,0.531,0.624,0.566,0.496,0.511,0.566,0.804,0.463,0.532,0.636,0.602,0.458
ACM+TS+SVM,0.608,62,0.690,0.640,0.493,0.762,0.617,0.574,0.500,0.549,0.692,0.563,0.537,0.610,0.556,0.560,0.766,0.652,0.578,0.476,0.516,0.659,0.577,0.532,0.530,0.705,0.686,0.690,0.609,0.667,0.851,0.627,0.424,0.558,0.610,0.511,0.577,0.685,0.833,0.824,0.833,0.563,0.511,0.560,0.571,0.489,0.559,0.700,0.519,0.567,0.523,0.839,0.463,0.706,0.720,0.625,0.580,0.500,0.799,0.566,0.500,0.464,0.545,0.533
ShallowConvNet,0.603,62,0.746,0.737,0.508,0.619,0.542,0.560,0.452,0.628,0.590,0.521,0.583,0.516,0.549,0.598,0.571,0.605,0.518,0.483,0.734,0.618,0.604,0.453,0.612,0.664,0.510,0.838,0.541,0.854,0.709,0.881,0.481,0.587,0.473,0.533,0.600,0.531,0.720,0.718,0.651,0.551,0.457,0.487,0.620,0.543,0.506,0.726,0.567,0.513,0.607,0.943,0.520,0.766,0.582,0.574,0.575,0.553,0.656,0.597,0.561,0.594,0.631,0.621
VAR+shLDA,0.602,62,0.535,0.720,0.547,0.669,0.567,0.654,0.548,0.690,0.738,0.597,0.488,0.551,0.556,0.552,0.474,0.623,0.506,0.439,0.767,0.648,0.588,0.436,0.608,0.705,0.597,0.738,0.563,0.737,0.631,0.672,0.483,0.543,0.676,0.524,0.628,0.575,0.771,0.833,0.490,0.568,0.538,0.479,0.616,0.550,0.603,0.614,0.673,0.490,0.650,0.854,0.520,0.751,0.660,0.631,0.469,0.514,0.741,0.487,0.594,0.445,0.578,0.616
Log-bandpower+shLDA,0.591,62,0.655,0.673,0.567,0.700,0.506,0.637,0.491,0.672,0.630,0.564,0.520,0.467,0.451,0.499,0.545,0.553,0.613,0.479,0.748,0.602,0.617,0.518,0.608,0.740,0.526,0.912,0.524,0.761,0.551,0.627,0.481,0.532,0.547,0.500,0.689,0.599,0.697,0.705,0.596,0.598,0.510,0.511,0.555,0.508,0.532,0.633,0.539,0.509,0.580,0.871,0.541,0.672,0.592,0.636,0.525,0.547,0.621,0.555,0.638,0.521,0.612,0.559
Microstates+GB,0.552,62,0.585,0.500,0.445,0.692,0.578,0.562,0.505,0.687,0.484,0.504,0.566,0.546,0.511,0.477,0.474,0.534,0.459,0.557,0.612,0.626,0.633,0.577,0.546,0.610,0.646,0.780,0.473,0.647,0.595,0.657,0.425,0.607,0.426,0.547,0.517,0.593,0.551,0.589,0.651,0.610,0.475,0.472,0.472,0.521,0.491,0.501,0.495,0.566,0.500,0.512,0.569,0.613,0.528,0.489,0.550,0.576,0.623,0.497,0.545,0.705,0.497,0.438


,MI_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20,sub-21,sub-22,sub-23,sub-24,sub-25,sub-26,sub-27,sub-28,sub-29,sub-30,sub-31,sub-32,sub-33,sub-34,sub-35,sub-36,sub-37,sub-38,sub-39,sub-40,sub-41,sub-42,sub-43,sub-44,sub-45,sub-46,sub-47,sub-48,sub-49,sub-50,sub-51,sub-52,sub-53,sub-54,sub-55,sub-56,sub-57,sub-58,sub-59,sub-60,sub-61,sub-62
method,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
FBCSP+ElasticNet,0.715,62,0.919,0.944,0.558,0.817,0.892,0.683,0.496,0.789,0.863,0.646,0.592,0.644,0.567,0.586,0.680,0.625,0.595,0.501,0.941,0.631,0.669,0.558,0.784,0.860,0.783,0.987,0.654,0.976,0.914,0.974,0.438,0.505,0.610,0.646,0.765,0.721,0.984,0.966,0.853,0.746,0.554,0.536,0.572,0.460,0.775,0.827,0.655,0.466,0.801,0.990,0.559,0.719,0.814,0.762,0.444,0.764,0.891,0.574,0.899,0.591,0.617,0.674
ACM+TS+SVM,0.688,62,0.793,0.718,0.563,0.913,0.667,0.662,0.526,0.672,0.877,0.672,0.542,0.665,0.575,0.633,0.804,0.715,0.597,0.493,0.918,0.664,0.738,0.641,0.610,0.803,0.779,0.953,0.655,0.855,0.900,0.903,0.432,0.541,0.655,0.454,0.655,0.754,0.921,0.908,0.987,0.635,0.503,0.535,0.705,0.470,0.691,0.767,0.535,0.551,0.706,0.980,0.500,0.774,0.780,0.692,0.674,0.690,0.914,0.612,0.372,0.522,0.592,0.617
EEGNet,0.678,62,0.870,0.926,0.393,0.847,0.739,0.700,0.526,0.897,0.806,0.582,0.537,0.603,0.511,0.561,0.386,0.597,0.492,0.572,0.880,0.883,0.736,0.493,0.885,0.823,0.730,0.955,0.501,0.998,0.892,0.941,0.561,0.602,0.544,0.683,0.779,0.688,0.919,0.818,0.949,0.677,0.531,0.443,0.566,0.564,0.602,0.852,0.539,0.690,0.513,0.983,0.481,0.653,0.584,0.532,0.531,0.677,0.915,0.470,0.599,0.637,0.683,0.502
ShallowConvNet,0.656,62,0.810,0.812,0.537,0.640,0.592,0.616,0.479,0.828,0.646,0.594,0.627,0.575,0.545,0.599,0.618,0.660,0.531,0.491,0.792,0.679,0.672,0.418,0.696,0.804,0.527,0.980,0.595,0.918,0.841,0.926,0.509,0.622,0.542,0.615,0.699,0.580,0.870,0.886,0.731,0.620,0.491,0.451,0.609,0.557,0.576,0.776,0.604,0.494,0.666,0.994,0.526,0.769,0.547,0.601,0.554,0.697,0.796,0.638,0.674,0.649,0.648,0.657
VAR+shLDA,0.649,62,0.591,0.809,0.581,0.767,0.522,0.756,0.531,0.758,0.736,0.604,0.462,0.544,0.597,0.572,0.477,0.670,0.546,0.439,0.835,0.773,0.574,0.441,0.676,0.783,0.656,0.818,0.629,0.921,0.662,0.826,0.336,0.533,0.708,0.617,0.726,0.613,0.857,0.929,0.583,0.636,0.617,0.484,0.664,0.556,0.620,0.655,0.677,0.546,0.710,0.927,0.516,0.720,0.716,0.687,0.512,0.602,0.840,0.511,0.816,0.478,0.623,0.660
Log-bandpower+shLDA,0.644,62,0.649,0.746,0.576,0.712,0.633,0.709,0.485,0.816,0.679,0.648,0.492,0.492,0.451,0.529,0.552,0.596,0.663,0.448,0.827,0.804,0.645,0.509,0.630,0.818,0.534,0.978,0.574,0.879,0.598,0.871,0.473,0.539,0.603,0.609,0.769,0.595,0.850,0.765,0.564,0.702,0.571,0.493,0.559,0.561,0.574,0.707,0.584,0.542,0.624,0.958,0.553,0.748,0.636,0.667,0.521,0.715,0.665,0.560,0.843,0.532,0.653,0.624
Microstates+GB,0.578,62,0.612,0.490,0.447,0.774,0.544,0.526,0.520,0.775,0.473,0.510,0.620,0.540,0.510,0.459,0.476,0.595,0.457,0.593,0.684,0.632,0.654,0.542,0.550,0.663,0.619,0.858,0.481,0.704,0.687,0.714,0.437,0.589,0.449,0.494,0.537,0.616,0.532,0.593,0.853,0.630,0.456,0.486,0.501,0.574,0.544,0.554,0.500,0.540,0.552,0.585,0.595,0.665,0.553,0.530,0.600,0.613,0.707,0.507,0.600,0.723,0.527,0.455


,OOD_AUROC,n,sub-01,sub-02,sub-03,sub-04,sub-05,sub-06,sub-07,sub-08,sub-09,sub-10,sub-11,sub-12,sub-13,sub-14,sub-15,sub-16,sub-17,sub-18,sub-19,sub-20,sub-21,sub-22,sub-23,sub-24,sub-25,sub-26,sub-27,sub-28,sub-29,sub-30,sub-31,sub-32,sub-33,sub-34,sub-35,sub-36,sub-37,sub-38,sub-39,sub-40,sub-41,sub-42,sub-43,sub-44,sub-45,sub-46,sub-47,sub-48,sub-49,sub-50,sub-51,sub-52,sub-53,sub-54,sub-55,sub-56,sub-57,sub-58,sub-59,sub-60,sub-61,sub-62
method,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
EEGNet+TempDens,0.606,62,0.610,0.871,0.608,0.555,0.692,0.549,0.414,0.627,0.464,0.493,0.610,0.710,0.715,0.876,0.570,0.526,0.753,0.557,0.485,0.644,0.567,0.459,0.853,0.535,0.635,0.563,0.491,0.820,0.616,0.819,0.672,0.386,0.543,0.407,0.440,0.765,0.792,0.806,0.697,0.319,0.608,0.874,0.472,0.698,0.717,0.765,0.377,0.355,0.576,0.839,0.396,0.465,0.642,0.814,0.521,0.455,0.746,0.430,0.531,0.714,0.521,0.518
FBCSP+IsolationForest,0.573,62,0.660,0.889,0.566,0.465,0.500,0.533,0.418,0.682,0.466,0.535,0.474,0.359,0.524,0.932,0.499,0.420,0.660,0.573,0.628,0.424,0.587,0.437,0.863,0.465,0.559,0.701,0.621,0.844,0.734,0.810,0.763,0.270,0.450,0.167,0.644,0.732,0.871,0.864,0.503,0.279,0.435,0.864,0.332,0.662,0.798,0.746,0.399,0.698,0.643,0.611,0.408,0.421,0.723,0.693,0.500,0.529,0.573,0.492,0.347,0.348,0.530,0.379
ACM+TS+OCSVM,0.556,62,0.667,0.614,0.540,0.415,0.539,0.461,0.174,0.646,0.723,0.424,0.658,0.376,0.475,0.958,0.441,0.702,0.752,0.572,0.572,0.488,0.415,0.463,0.928,0.394,0.488,0.603,0.318,0.786,0.631,0.668,0.507,0.432,0.593,0.200,0.425,0.692,0.820,0.678,0.603,0.273,0.625,0.914,0.618,0.524,0.543,0.532,0.538,0.521,0.514,0.549,0.347,0.532,0.414,0.900,0.437,0.562,0.675,0.503,0.582,0.692,0.461,0.389
VAR+Mahalanobis,0.539,62,0.567,0.467,0.566,0.464,0.376,0.383,0.475,0.762,0.578,0.453,0.597,0.352,0.507,0.952,0.415,0.589,0.770,0.659,0.609,0.552,0.425,0.412,0.839,0.543,0.498,0.408,0.420,0.861,0.612,0.504,0.564,0.434,0.529,0.356,0.444,0.671,0.717,0.523,0.713,0.328,0.568,0.739,0.525,0.543,0.565,0.614,0.597,0.477,0.507,0.452,0.266,0.497,0.442,0.823,0.524,0.663,0.562,0.459,0.173,0.642,0.478,0.418
ShallowConvNet+kNN,0.535,62,0.490,0.784,0.552,0.538,0.707,0.569,0.383,0.646,0.515,0.514,0.455,0.569,0.584,0.537,0.516,0.408,0.249,0.480,0.477,0.585,0.420,0.360,0.253,0.505,0.488,0.790,0.581,0.399,0.598,0.732,0.601,0.346,0.454,0.457,0.467,0.719,0.239,0.617,0.690,0.549,0.599,0.770,0.407,0.792,0.692,0.379,0.447,0.322,0.631,0.787,0.489,0.435,0.681,0.477,0.653,0.522,0.569,0.433,0.505,0.800,0.409,0.520
Log-bandpower+Mahalanobis,0.528,62,0.609,0.683,0.484,0.371,0.549,0.480,0.434,0.604,0.428,0.498,0.539,0.246,0.461,0.895,0.494,0.462,0.611,0.515,0.555,0.461,0.508,0.497,0.801,0.425,0.620,0.326,0.450,0.783,0.691,0.580,0.567,0.330,0.486,0.339,0.555,0.593,0.621,0.673,0.490,0.272,0.549,0.790,0.528,0.519,0.448,0.533,0.535,0.668,0.452,0.527,0.374,0.536,0.436,0.878,0.493,0.561,0.488,0.501,0.433,0.535,0.593,0.401
Microstates+LOF,0.507,62,0.517,0.640,0.539,0.519,0.451,0.528,0.420,0.572,0.446,0.483,0.623,0.245,0.611,0.910,0.347,0.485,0.816,0.480,0.564,0.352,0.408,0.425,0.691,0.490,0.531,0.524,0.372,0.584,0.567,0.495,0.453,0.253,0.562,0.369,0.410,0.321,0.510,0.559,0.680,0.312,0.600,0.868,0.625,0.483,0.426,0.391,0.511,0.427,0.479,0.542,0.334,0.521,0.520,0.769,0.459,0.510,0.526,0.468,0.387,0.820,0.361,0.337


In [5]:
display_summary_by_dataset(
    experiment,
    MI_METHODS,
    OOD_METHODS,
    datasets=datasets,
    experiment_specs=[
        ("MI BA", "mi", "MI_BA"),
        ("MI AUROC", "mi", "MI_AUROC"),
        ("OOD", "ood", "OOD_AUROC"),
    ],
)
